## Results require recomputation after the 2026-11-06 feature correction

The price-adjustment timing has been corrected and the feature table rebuilt. Saved outputs were cleared and prior models, metrics and model-selection results were preserved under archive/price_adjustment_fix_20261106. Rerun notebook 03 and training/validation in 04 before interpreting results or using 05. Any previously written numerical conclusions require review. 2021 was evaluated earlier; repeating it is a reevaluation, not a new unseen test.


## HT2026-DA631E-TS056
## Mini Project 1
### Author: Nancy Truong, Argtim Zeqiraj

I divided the labeled historical data chronologically.
Model selection also uses expanding-window validation: training on 2017 and validating on 2018, training on 2017–2018 and validating on 2019, and training on 2017–2019 and validating on 2020. The final two trading dates are removed from each training window because their targets depend on future prices that could cross the evaluation boundary. The final model will be refitted on development data (2017-2020) before evaluation on the held-out test period (2021).

The main preprocessing steps are:

- Data-quality checks: I checked for duplicate stock-date observations, invalid prices, negative trading volumes, and missing identifiers.

- No-trade observations: The price data contains 7608 no-trade observations. These were retained and represented using trading-status indicators. A separate CloseForFeatures column uses the last available closing price for each stock through forward filling.

- Feature engineering: Returns, momentum, volatility, and trading-activity features are calculated within each stock’s chronological history. Price adjustments are handled using information available at the relevant date.

- Financial disclosure timing: Financial reports are joined to stock observations using their availability date, with a one-day disclosure lag. Each observation receives only a report available on or before its prediction date.

- Missing targets: Observations without Target are retained during data preparation but excluded from supervised training and evaluation.

- Model preprocessing: Numeric imputation and categorical encoding are fitted separately within each training fold. Ridge additionally uses numeric standardization and one-hot encoding. HGB uses categorical encoding with its categorical features explicitly identified.

The original stock-price file contains 2,332,531 observations and 12 raw columns, including identifiers and the target. After excluding observations with missing targets and purging dates at the split boundaries, 2,324,361 observations remain across the training, validation, and testing sets.

The periods do not overlap, and all three sets contain known Target values. The final two dates were removed from both the training and validation periods because their targets could depend on prices beyond the relevant boundary. These counts describe the initial chronological splits; the final model’s training size is reported separately when it is refitted on development data.

In [ ]:
#!pip install pandas numpy matplotlib missingno
#%pip install scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import missingno as msno

import os
from pathlib import Path

#import sklearn.preprocessing

pd.options.display.max_rows=1000

import sys
import json
from IPython.display import display


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the parent directory containing src/data.py."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data.py").is_file():
            return candidate

    raise FileNotFoundError("Could not find src/data.py")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
    


#from data import load_data, clean_prices, clean_stock_list, clean_financials
from config import ProjectConfig, ProjectPaths
from modeling import (
    build_model_pipeline
    ,chronological_split
    ,make_xy
    ,save_json
    ,select_recent_dates,
)
from features import (
    add_price_features
    ,assert_causal_feature_stability
    ,feature_missingness
    ,infer_feature_columns
    ,join_financials_asof
    ,join_stock_metadata
)

from modeling import save_json

config = ProjectConfig()
paths = ProjectPaths(PROJECT_ROOT)
paths.ensure_output_dirs

print("Project root:", PROJECT_ROOT)
print("Imported load_data successfully")

print("Imported libraries sucessfully! yay")

In [ ]:
features_table = pd.read_parquet(path=paths.processed / "features_table.parquet")
numeric_features, categorical_features = infer_feature_columns(features_table)

## 1. Construct supervised chronological splits

In [ ]:
splits, split_report = chronological_split(features_table=features_table, config=config)
train = splits['train']
valid = splits['validation']
test = splits['test']

display(split_report)
print("Purged from train: ", split_report.attrs['train_purged_dates'])
print("Purged from valid: ", split_report.attrs['validation_purged_dates'])

assert train["Date"].max() < valid["Date"].min()
assert valid["Date"].max() < test["Date"].min()
assert train["Target"].notna().all()
assert valid["Target"].notna().all()
assert test["Target"].notna().all()

In [ ]:
train.isna().sum()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.4))
colors = {"train": "#4C78A8", "validation": "#F2CF5B", "test": "#E45756"}

for row, (name, frame) in enumerate(splits.items()):
    ax.barh(0, (frame["Date"].max() - frame["Date"].min()).days,
            left=frame["Date"].min(), height=0.5, color=colors[name], label=name)
ax.set_yticks([])
ax.set_title("Chronological experiment design")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.25))
plt.show()

## Make X and y

In [ ]:
X_train, y_train = make_xy(df=train, numeric_features=numeric_features, categorical_features=categorical_features)
X_valid, y_valid = make_xy(valid, numeric_features, categorical_features)

matrix_report = pd.DataFrame({
    "object": ["X_train", "y_train", "X_valid", "y_valid"],
    "rows": [len(X_train), len(y_train), len(X_valid), len(y_valid)],
    "columns": [X_train.shape[1], 1, X_valid.shape[1], 1],
})
display(matrix_report)
assert list(X_train.columns) == list(X_valid.columns)

## fit_transform and transform

In [ ]:
ridge_pipeline = build_model_pipeline(
    "ridge"
    ,numeric_features
    ,categorical_features
)
ridge_pipeline

## Small smoke test

Fit on the most recent 20 complete training dates and predict the first two
validation dates.

In [ ]:
smoke_train = select_recent_dates(train, max_dates=20)
valid_dates = valid['Date'].drop_duplicates().sort_values().iloc[:2]
smoke_valid = valid.loc[valid['Date'].isin(valid_dates)]

X_smoke_train, y_smoke_train = make_xy(smoke_train, numeric_features, categorical_features)
X_smoke_valid, y_valid_smoke = make_xy(smoke_valid, numeric_features, categorical_features)

smoke_pipeline = build_model_pipeline("ridge", numeric_features, categorical_features)
smoke_pipeline.fit(X_smoke_train, y_smoke_train)
smoke_predictions = smoke_pipeline.predict(X_smoke_valid)

assert len(smoke_predictions) == len(smoke_valid)
assert np.isfinite(smoke_predictions).all()
print(f"Smoke test passed for {len(smoke_predictions):,} validation rows")

In [ ]:
transformed_names = smoke_pipeline.named_steps['preprocess'].get_feature_names_out()
display(pd.Series(transformed_names, name="transformed_features").head(50).to_frame())
print(f"Raw input: {X_smoke_train.shape[1]}; transformed columns: {len(transformed_names)}")

### save

In [ ]:
experiment_audit = {
    "split_report": split_report.reset_index().to_dict(orient="records"),
    "train_purged_dates": split_report.attrs["train_purged_dates"],
    "validation_purged_dates": split_report.attrs["validation_purged_dates"],
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
}
save_json(experiment_audit, paths.reports / "experiment_audit.json")
print("Saved reports/experiment_audit.json")